# MovieLens 20M — EDA, Data Cleaning & Feature Engineering

**Project:** Movie Recommender via Neural Collaborative Filtering (NCF / NeuMF)

This notebook prepares MovieLens 20M for the project described in the PRD.

### Important dataset alignment
The provided Kaggle dataset is **MovieLens 20M**, while the PRD mentions MovieLens 25M. This notebook is intentionally written for **MovieLens 20M**. Keep the dataset version consistent across experiments, reports and the final viva.

### Goals
1. Load and validate the MovieLens files.
2. Perform systematic EDA without making unsupported assumptions.
3. Clean and validate ratings/movie metadata.
4. Measure sparsity, user activity and movie activity.
5. Re-index user/movie IDs for neural embeddings.
6. Create explicit and implicit-feedback signals.
7. Prepare negative-sampling inputs.
8. Create a leave-one-out evaluation split.
9. Save reusable processed artifacts for NCF training.

**No model training is performed here.**

## 0. Environment and configuration

The notebook supports either:
- a manually downloaded/unzipped MovieLens 20M folder, or
- a Kaggle download if the Kaggle CLI is configured.

For a local setup, place the extracted files under `data/ml-20m/`.

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Folders
DATA_ROOT = Path("data")
DATA_DIR = DATA_ROOT / "ml-20m"        # extracted MovieLens 20M files
OUTPUT_DIR = Path("data/processed")    # processed files will be saved here

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder  :", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

## 1. Load the core files

For the core NCF pipeline, `ratings.csv` and `movies.csv` are required.

Optional files:
- `tags.csv`
- `links.csv`
- `genome-scores.csv`
- `genome-tags.csv`

We load the optional files only if they exist.

In [ ]:
import urllib.request
import zipfile

ML20M_URL = "https://files.grouplens.org/datasets/movielens/ml-20m.zip"
zip_path = DATA_ROOT / "ml-20m.zip"

files_exist = (DATA_DIR / "ratings.csv").exists() and (DATA_DIR / "movies.csv").exists()

if files_exist:
    print("Dataset already present, skipping download.")
else:
    if not zip_path.exists():
        print("Downloading MovieLens 20M (~190 MB)...")
        urllib.request.urlretrieve(ML20M_URL, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)
    print("Done.")

# Stop early if a required file is missing
for name in ["ratings.csv", "movies.csv"]:
    assert (DATA_DIR / name).exists(), f"Missing {DATA_DIR / name}"

print("Files found:", sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
# Smaller dtypes = less memory for 20M rows
ratings = pd.read_csv(
    DATA_DIR / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)
movies = pd.read_csv(DATA_DIR / "movies.csv")

# tags.csv is optional
tags = None
if (DATA_DIR / "tags.csv").exists():
    tags = pd.read_csv(DATA_DIR / "tags.csv")
    print("Tags:", tags.shape)

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)

## 2. Basic structural EDA

Check:
- dimensions
- columns
- dtypes
- sample rows
- unique users/movies
- rating range
- timestamp range

These checks establish what is actually present before cleaning.

In [ ]:
display(ratings.head())
display(movies.head())

print("Ratings info:")
ratings.info()

print("Movies info:")
movies.info()

print("Unique users:", ratings["userId"].nunique())
print("Unique movies in ratings:", ratings["movieId"].nunique())
print("Movies in movies.csv:", movies["movieId"].nunique())
print("Rating min/max:", ratings["rating"].min(), ratings["rating"].max())
print("Rating unique values:", sorted(ratings["rating"].unique()))

## 3. Missing values and duplicates

Do not delete rows automatically. First quantify the problem and inspect it.

We check:
- missing values
- exact duplicate rows
- repeated `(userId, movieId)` pairs
- orphan movie IDs

In [ ]:
# Missing values
for name, df in [("ratings", ratings), ("movies", movies)]:
    print("Missing values in", name)
    display(pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(4),
    }))

# Duplicates
print("Exact duplicate rating rows:", ratings.duplicated().sum())
print("Repeated user-movie pairs:", ratings.duplicated(["userId", "movieId"]).sum())

# Orphan movie IDs (rated, but not in movies.csv)
orphan_movie_ids = set(ratings["movieId"]) - set(movies["movieId"])
print("Movie IDs in ratings but absent from movies.csv:", len(orphan_movie_ids))

## 4. Rating distribution and interaction statistics

The project uses explicit ratings, but the main recommendation task will also use an implicit-feedback representation.

We calculate:
- count of each rating
- percentage of each rating
- average rating
- median rating
- standard deviation

In [ ]:
rating_summary = ratings["rating"].value_counts().sort_index().reset_index()
rating_summary.columns = ["rating", "count"]
rating_summary["percentage"] = rating_summary["count"] / len(ratings) * 100

display(rating_summary)
print("Mean rating  :", round(ratings["rating"].mean(), 4))
print("Median rating:", ratings["rating"].median())
print("Rating std   :", round(ratings["rating"].std(), 4))

## 5. User activity and movie activity

These distributions are central to recommender-system analysis.

We measure:
- number of ratings per user
- number of ratings per movie
- average rating per user
- average rating per movie
- minimum/median/maximum activity

These measurements reveal sparsity, highly active users, popular movies and long-tail behavior.

In [ ]:
# Per-user and per-movie statistics
user_stats = ratings.groupby("userId")["rating"].agg(["count", "mean", "std"]).reset_index()
user_stats.columns = ["userId", "rating_count", "avg_rating", "rating_std"]

movie_stats = ratings.groupby("movieId")["rating"].agg(["count", "mean", "std"]).reset_index()
movie_stats.columns = ["movieId", "rating_count", "avg_rating", "rating_std"]

print("User activity:")
display(user_stats["rating_count"].describe())

print("Movie activity:")
display(movie_stats["rating_count"].describe())

# Top 10 most-rated movies
top_movies = movie_stats.sort_values("rating_count", ascending=False).head(10)
top_movies = top_movies.merge(movies[["movieId", "title"]], on="movieId", how="left")

print("Top 10 movies by number of ratings:")
display(top_movies[["movieId", "title", "rating_count", "avg_rating"]])

## 6. Rating confidence / popularity caveat

A movie with a very high average rating but only a few ratings is not directly comparable with a movie having thousands of ratings.

For EDA, show both:
- raw average rating
- number of ratings

For optional reporting, use a minimum-rating threshold or a Bayesian/shrinkage score rather than claiming that raw average alone means "best movie".

In [ ]:
MIN_RATINGS_FOR_SUMMARY = 100

qualified_movies = movie_stats[movie_stats["rating_count"] >= MIN_RATINGS_FOR_SUMMARY]
print(f"Movies with at least {MIN_RATINGS_FOR_SUMMARY} ratings:", len(qualified_movies))

best_movies = qualified_movies.sort_values(["avg_rating", "rating_count"], ascending=False).head(20)
best_movies = best_movies.merge(movies[["movieId", "title"]], on="movieId", how="left")

display(best_movies[["movieId", "title", "rating_count", "avg_rating"]])

## 7. Genre processing

MovieLens stores genres as pipe-separated strings such as:

`Action|Adventure|Sci-Fi`

For the hybrid/content-based extension, convert these into:
- a list of genres
- multi-hot genre columns
- genre counts

The pure NCF model does not require these columns, but they are useful for EDA and later hybrid re-ranking.

In [ ]:
movies_clean = movies.copy()

movies_clean["genres"] = movies_clean["genres"].fillna("(no genres listed)")
movies_clean["genre_list"] = movies_clean["genres"].str.split("|")

# How many movies per genre
genre_counts = movies_clean["genre_list"].explode().value_counts().reset_index()
genre_counts.columns = ["genre", "movie_count"]
all_genres = sorted(genre_counts["genre"])

print("Number of unique genre labels:", len(all_genres))
print(all_genres)
display(genre_counts)

## 8. Extract movie year from title

Movie titles commonly contain a year such as `Toy Story (1995)`.

We extract the final four-digit year when available. Missing years remain null.

This is a metadata feature for analysis/hybrid modeling, not a required input to pure NeuMF.

In [ ]:
# Year is the last "(1995)" in the title
movies_clean["release_year"] = movies_clean["title"].str.extract(r"\((\d{4})\)\s*$", expand=False)
movies_clean["release_year"] = movies_clean["release_year"].astype("Int64")

print("Movies with extracted year   :", movies_clean["release_year"].notna().sum())
print("Movies without extracted year:", movies_clean["release_year"].isna().sum())
display(movies_clean[["movieId", "title", "release_year"]].head(10))

## 9. Timestamp conversion

MovieLens timestamps are Unix timestamps.

Keep the original timestamp for reproducibility and create a datetime column for analysis and possible temporal experiments.

In [ ]:
ratings_clean = ratings.copy()

ratings_clean["rating_datetime"] = pd.to_datetime(
    ratings_clean["timestamp"], unit="s", errors="coerce", utc=True
)

print("Invalid timestamps:", ratings_clean["rating_datetime"].isna().sum())
display(ratings_clean.head())

## 10. Core data cleaning

Cleaning rules used here are deliberately conservative:
1. Keep only rows with valid user/movie/rating/timestamp fields.
2. Keep ratings within the MovieLens 0.5–5.0 scale.
3. Remove exact duplicate rows.
4. Remove rating rows whose movie does not exist in `movies.csv`.
5. Do not automatically delete low-activity users/movies; make that an explicit modeling experiment.

This prevents accidental information loss.

In [ ]:
rows_before = len(ratings_clean)

# 1. Drop rows with missing values
ratings_clean = ratings_clean.dropna(subset=["userId", "movieId", "rating", "timestamp"])

# 2. Keep ratings on the 0.5 - 5.0 scale
ratings_clean = ratings_clean[ratings_clean["rating"].between(0.5, 5.0)]

# 3. Remove exact duplicate rows
ratings_clean = ratings_clean.drop_duplicates()

# 4. Remove ratings for movies that are not in movies.csv
ratings_clean = ratings_clean[ratings_clean["movieId"].isin(movies_clean["movieId"])].copy()

rows_after = len(ratings_clean)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning :", rows_after)
print("Rows removed        :", rows_before - rows_after)
print("Duplicate rows left :", ratings_clean.duplicated().sum())
print("Invalid ratings left:", (~ratings_clean["rating"].between(0.5, 5.0)).sum())

## 11. Optional low-activity filtering

For a large NCF experiment, you may choose a minimum number of interactions per user/movie.

**Important:** filtering changes the population and therefore changes your experiment. Record the thresholds in your experiment configuration.

The default below is intentionally configurable rather than silently applied.

In [ ]:
MIN_USER_RATINGS = 5
MIN_MOVIE_RATINGS = 5

# How many ratings each user / movie has
user_counts = ratings_clean["userId"].value_counts()
movie_counts = ratings_clean["movieId"].value_counts()

# Keep only users / movies with enough ratings
good_users = user_counts[user_counts >= MIN_USER_RATINGS].index
good_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index

keep = ratings_clean["userId"].isin(good_users) & ratings_clean["movieId"].isin(good_movies)
ratings_filtered = ratings_clean[keep].copy()

print("Before activity filtering:", ratings_clean.shape)
print("After activity filtering :", ratings_filtered.shape)
print("Users retained :", ratings_filtered["userId"].nunique())
print("Movies retained:", ratings_filtered["movieId"].nunique())

## 12. Re-index users and movies for embeddings

Raw IDs are identifiers, not continuous embedding indices.

Create:
- `user_idx` in `[0, num_users-1]`
- `movie_idx` in `[0, num_movies-1]`

Keep mapping tables so predictions can be translated back to original MovieLens IDs.

In [ ]:
ratings_model = ratings_filtered.copy()

# Category codes are 0..N-1 in sorted ID order
ratings_model["user_idx"] = ratings_model["userId"].astype("category").cat.codes.astype("int64")
ratings_model["movie_idx"] = ratings_model["movieId"].astype("category").cat.codes.astype("int64")

# Mapping tables (to convert back to the original MovieLens IDs)
user_mapping = ratings_model[["userId", "user_idx"]].drop_duplicates().sort_values("user_idx")
user_mapping = user_mapping.reset_index(drop=True)

movie_mapping = ratings_model[["movieId", "movie_idx"]].drop_duplicates().sort_values("movie_idx")
movie_mapping = movie_mapping.reset_index(drop=True)

print("Number of users :", len(user_mapping))
print("Number of movies:", len(movie_mapping))
display(ratings_model.head())

## 13. Explicit and implicit feedback features

Keep the original rating for rating prediction.

For the recommendation/classification view, create a configurable positive-interaction threshold.

Example:
- `rating >= 4.0` → positive interaction
- otherwise → non-positive

Do not interpret an unrated movie as automatically disliked. Unrated items will be sampled separately as candidate negatives for training.

In [ ]:
POSITIVE_RATING_THRESHOLD = 4.0

# 1 = liked (rating >= 4.0), 0 = not liked
ratings_model["is_positive"] = (ratings_model["rating"] >= POSITIVE_RATING_THRESHOLD).astype("int8")

print(ratings_model["is_positive"].value_counts())
print("Positive interaction rate:", round(ratings_model["is_positive"].mean() * 100, 3), "%")

## 14. Negative-sampling candidate preparation

For each user, define:
- positive/observed movie set
- unseen movie set

Do not materialize every user × movie pair: the matrix would be enormous.

Instead, store the observed interaction sets and sample unseen movie IDs on demand during model training.

In [ ]:
NUM_USERS = len(user_mapping)
NUM_MOVIES = len(movie_mapping)

# For every user: the set of movies they have rated
user_seen_items = ratings_model.groupby("user_idx")["movie_idx"].apply(set).to_dict()

print("Users with interaction sets:", len(user_seen_items))
print("Total user-item matrix cells if dense:", NUM_USERS * NUM_MOVIES)
print("Observed interactions:", len(ratings_model))


# Pick random movies the user has NOT rated
def sample_negative_items(user_idx, n_negatives=4):
    seen = user_seen_items[int(user_idx)]
    negatives = []
    while len(negatives) < n_negatives:
        movie = random.randrange(NUM_MOVIES)
        if movie not in seen:
            negatives.append(movie)
    return negatives


example_user = int(ratings_model["user_idx"].iloc[0])
print("Example user:", example_user)
print("Sample negatives:", sample_negative_items(example_user, 5))

## 15. Leave-one-out split

For recommendation evaluation, the PRD calls for a leave-one-out protocol.

A practical chronological version:
- sort each user's interactions by timestamp
- last interaction → test
- second-last interaction → validation
- earlier interactions → train

For the main recommendation evaluation, you should decide whether the held-out interaction is:
- any rating, or
- only a positive interaction.

The code below uses the last chronological interaction and keeps the original rating for later analysis.

In [ ]:
# Sort each user's ratings by time
ratings_model = ratings_model.sort_values(["user_idx", "timestamp"])

# 0 = user's last rating, 1 = second-last, 2 = older ...
position_from_end = ratings_model.groupby("user_idx").cumcount(ascending=False)

test_df = ratings_model[position_from_end == 0].copy()    # last interaction
val_df = ratings_model[position_from_end == 1].copy()     # second-last interaction
train_df = ratings_model[position_from_end >= 2].copy()   # everything earlier

print("Train     :", train_df.shape)
print("Validation:", val_df.shape)
print("Test      :", test_df.shape)

print("Users in train     :", train_df["user_idx"].nunique())
print("Users in validation:", val_df["user_idx"].nunique())
print("Users in test      :", test_df["user_idx"].nunique())

## 16. Feature tables for later hybrid re-ranking

Create movie metadata features:
- movie ID
- title
- release year
- genre multi-hot columns

Also calculate popularity statistics from **training data only** to avoid leakage into evaluation.

In [ ]:
# Genre multi-hot columns (one 0/1 column per genre)
genre_dummies = movies_clean["genres"].str.get_dummies(sep="|").astype("int8")

movies_features = pd.concat(
    [movies_clean[["movieId", "title", "release_year"]], genre_dummies],
    axis=1,
)

# Popularity from TRAIN data only (so test data does not leak in)
train_movie_stats = train_df.groupby("movieId")["rating"].agg(["count", "mean"]).reset_index()
train_movie_stats.columns = ["movieId", "train_rating_count", "train_avg_rating"]

movies_features = movies_features.merge(train_movie_stats, on="movieId", how="left")
movies_features["train_rating_count"] = movies_features["train_rating_count"].fillna(0).astype("int32")

print("Movie feature table shape:", movies_features.shape)
display(movies_features.head())

## 17. Data-quality assertions

These assertions make the preprocessing pipeline reproducible and safer.

In [ ]:
assert ratings_model["user_idx"].min() >= 0
assert ratings_model["movie_idx"].min() >= 0
assert ratings_model["user_idx"].max() < NUM_USERS
assert ratings_model["movie_idx"].max() < NUM_MOVIES
assert ratings_model["rating"].between(0.5, 5.0).all()
assert train_df["timestamp"].max() <= test_df["timestamp"].max()

print("All core data-quality assertions passed.")

## 18. Save processed artifacts

The outputs are designed for later PyTorch/FastAPI work.

Files:
- `ratings_model.csv`
- `train.csv`
- `validation.csv`
- `test.csv`
- `user_mapping.csv`
- `movie_mapping.csv`
- `movies_features.csv`
- `preprocessing_config.json`

For a 20M-row dataset, Parquet is recommended for repeated local experimentation if `pyarrow` is installed.

In [ ]:
ratings_model.to_csv(OUTPUT_DIR / "ratings_model.csv", index=False)
train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "validation.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

user_mapping.to_csv(OUTPUT_DIR / "user_mapping.csv", index=False)
movie_mapping.to_csv(OUTPUT_DIR / "movie_mapping.csv", index=False)
movies_features.to_csv(OUTPUT_DIR / "movies_features.csv", index=False)

config = {
    "seed": SEED,
    "dataset": "MovieLens 20M",
    "min_user_ratings": MIN_USER_RATINGS,
    "min_movie_ratings": MIN_MOVIE_RATINGS,
    "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
    "split": "chronological_leave_one_out",
    "notes": "Adjust thresholds as an explicit experiment; do not silently change them.",
}

with open(OUTPUT_DIR / "preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved processed artifacts to:", OUTPUT_DIR.resolve())

## 19. Baseline evaluation setup

Next we build two baselines before NeuMF:
1. **Popularity-based ranking** — a non-personalized fallback.
2. **Implicit Matrix Factorization (MF)** — user and movie embeddings trained to rank positive ratings above sampled unseen movies.

For ranking metrics, a held-out test interaction is considered relevant only when `rating >= POSITIVE_RATING_THRESHOLD`. We evaluate a candidate set containing that positive movie and sampled unseen movies. The first run evaluates a reproducible sample of users so it remains practical in Colab; increase `MAX_EVAL_USERS` or set it to `None` for a larger evaluation.

Important: this follows the implicit-feedback ranking objective used by NeuMF. It is not the same as explicit rating prediction; RMSE can be added separately if needed.

In [ ]:
# Settings (start small for Colab, then scale up)
MAX_EVAL_USERS = 10_000            # None = evaluate all users
N_EVAL_NEGATIVES = 99              # 1 positive movie + 99 random unseen movies
MAX_POSITIVE_TRAIN_ROWS = 500_000  # None = use all positive training rows
MF_EMBEDDING_DIM = 32
MF_EPOCHS = 3
MF_BATCH_SIZE = 4096
MF_LEARNING_RATE = 1e-3

# Movies each user has rated (train + val + test), so we never pick them as negatives
all_seen_by_user = user_seen_items

# Popularity statistics from TRAIN data only
GLOBAL_MEAN = float(train_df["rating"].mean())
SHRINKAGE_M = 100

movie_popularity = train_df.groupby("movie_idx")["rating"].agg(["count", "mean"])
movie_popularity.columns = ["train_rating_count", "train_avg_rating"]

# Make sure every movie has a row (movies not in train get count 0)
movie_popularity = movie_popularity.reindex(range(NUM_MOVIES))
movie_popularity.index.name = "movie_idx"
movie_popularity = movie_popularity.reset_index()
movie_popularity["train_rating_count"] = movie_popularity["train_rating_count"].fillna(0)
movie_popularity["train_avg_rating"] = movie_popularity["train_avg_rating"].fillna(GLOBAL_MEAN)

# Bayesian score: pulls movies with few ratings towards the global mean
n = movie_popularity["train_rating_count"]
avg = movie_popularity["train_avg_rating"]
movie_popularity["bayesian_score"] = (n * avg + SHRINKAGE_M * GLOBAL_MEAN) / (n + SHRINKAGE_M)

print("Users:", NUM_USERS, "Movies:", NUM_MOVIES)
print("Positive threshold:", POSITIVE_RATING_THRESHOLD)
print("Training positive interactions:", int((train_df["rating"] >= POSITIVE_RATING_THRESHOLD).sum()))
display(movie_popularity.sort_values(["train_rating_count", "bayesian_score"], ascending=False).head(10))

## 20. Popularity-based recommender

This recommender ranks movies by training-set rating count. The Bayesian score is also retained as an alternative ranking that accounts for rating quality and small sample sizes. Popularity is not personalized, but it is useful as a baseline and as a fallback for new users.

In [ ]:
# Rank by rating count (Bayesian score breaks ties)
popularity_ranked = movie_popularity.sort_values(
    ["train_rating_count", "bayesian_score"], ascending=False
).reset_index(drop=True)

popularity_score = dict(zip(movie_popularity["movie_idx"], movie_popularity["train_rating_count"]))

# movie_idx -> title
movie_titles = movie_mapping.merge(movies_clean[["movieId", "title"]], on="movieId", how="left")
movie_title_by_idx = dict(zip(movie_titles["movie_idx"], movie_titles["title"]))


# Top-k popular movies that the user has not rated yet
def recommend_popular_for_user(user_idx, k=10):
    seen = all_seen_by_user[int(user_idx)]
    unseen = popularity_ranked[~popularity_ranked["movie_idx"].isin(seen)]
    top = unseen.head(k).copy()
    top["title"] = top["movie_idx"].map(movie_title_by_idx)
    return top[["movie_idx", "title", "train_rating_count", "train_avg_rating", "bayesian_score"]].reset_index(drop=True)


example_user_idx = int(train_df["user_idx"].iloc[0])
print("Example user_idx:", example_user_idx)
display(recommend_popular_for_user(example_user_idx, k=10))

## 21. Ranking evaluation helpers (HR@10 and NDCG@10)

For each eligible user, the candidate list contains one held-out positive movie plus sampled movies the user has never rated. `HR@10` is 1 when the relevant movie appears in the top 10. `NDCG@10` rewards placing it nearer the top. Use exactly the same candidate sets for popularity, MF and NeuMF comparisons.

In [ ]:
# Users whose last (test) rating is positive
test_positive = test_df[test_df["rating"] >= POSITIVE_RATING_THRESHOLD][["user_idx", "movie_idx", "rating"]]

if MAX_EVAL_USERS is not None and len(test_positive) > MAX_EVAL_USERS:
    test_positive = test_positive.sample(n=MAX_EVAL_USERS, random_state=SEED)


# For each user: [positive movie] + 99 random movies they have not rated
def make_eval_candidates(test_positive_df, n_negatives=99, seed=42):
    rng = np.random.default_rng(seed)
    rows = []

    for user, positive_movie in zip(test_positive_df["user_idx"], test_positive_df["movie_idx"]):
        user = int(user)
        positive_movie = int(positive_movie)
        seen = all_seen_by_user[user]

        negatives = []
        while len(negatives) < n_negatives:
            movie = int(rng.integers(0, NUM_MOVIES))
            if movie not in seen and movie not in negatives:
                negatives.append(movie)

        rows.append((user, positive_movie, [positive_movie] + negatives))

    return rows


eval_candidates = make_eval_candidates(test_positive, N_EVAL_NEGATIVES, SEED)
print("Eligible test users evaluated:", len(eval_candidates))
print("Candidates per user:", 1 + N_EVAL_NEGATIVES)


# HR@k and NDCG@k for any scoring function score_fn(user, candidate_movies)
def ranking_metrics_from_scores(eval_rows, score_fn, k=10):
    hits = []
    ndcgs = []

    for user, positive_movie, candidates in eval_rows:
        candidates = np.array(candidates)
        scores = np.array(score_fn(user, candidates), dtype=float)

        # candidates sorted from best to worst score
        top_k = candidates[np.argsort(-scores)][:k]

        if positive_movie in top_k:
            rank = list(top_k).index(positive_movie)   # 0 = first place
            hits.append(1)
            ndcgs.append(1 / np.log2(rank + 2))
        else:
            hits.append(0)
            ndcgs.append(0)

    return {
        "HR@10": float(np.mean(hits)),
        "NDCG@10": float(np.mean(ndcgs)),
        "evaluated_users": len(eval_rows),
    }


# Popularity baseline: score = how many ratings the movie has
def popularity_scorer(user, candidates):
    return [popularity_score.get(int(m), 0) for m in candidates]


pop_metrics = ranking_metrics_from_scores(eval_candidates, popularity_scorer, k=10)
print("Popularity baseline metrics:", pop_metrics)

## 22. Implicit Matrix Factorization baseline (PyTorch)

This model learns one embedding vector per user and movie. Their dot product is the preference score. We train it with binary cross-entropy: positive-rated training interactions are label `1`, and sampled unobserved movies are label `0`.

The first run intentionally caps positive training rows for speed. Once the pipeline works, set `MAX_POSITIVE_TRAIN_ROWS = None`, increase epochs, and compare results. This model is a simple matrix-factorization baseline; NeuMF will later replace the fixed dot-product interaction with GMF + MLP branches.

In [ ]:
import torch
from torch import nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)

# Positive (user, movie) pairs from the training data
positive_train_df = train_df[train_df["rating"] >= POSITIVE_RATING_THRESHOLD][["user_idx", "movie_idx"]]
positive_train_df = positive_train_df.drop_duplicates()

if MAX_POSITIVE_TRAIN_ROWS is not None and len(positive_train_df) > MAX_POSITIVE_TRAIN_ROWS:
    positive_train_df = positive_train_df.sample(n=MAX_POSITIVE_TRAIN_ROWS, random_state=SEED)

train_users = positive_train_df["user_idx"].to_numpy()
train_movies = positive_train_df["movie_idx"].to_numpy()


# Matrix Factorization: score = user vector . movie vector + biases
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.movie_emb = nn.Embedding(n_movies, dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.movie_emb.weight, std=0.05)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)

    def forward(self, users, movies):
        dot = (self.user_emb(users) * self.movie_emb(movies)).sum(dim=1)
        bias = self.user_bias(users).squeeze(1) + self.movie_bias(movies).squeeze(1)
        return dot + bias + self.global_bias


# One random unseen movie for every training pair
rng_train = np.random.default_rng(SEED)

def sample_one_negative_per_user(users):
    negatives = rng_train.integers(0, NUM_MOVIES, size=len(users))
    for i, user in enumerate(users):
        while int(negatives[i]) in all_seen_by_user[int(user)]:
            negatives[i] = rng_train.integers(0, NUM_MOVIES)
    return negatives


mf_model = MatrixFactorization(NUM_USERS, NUM_MOVIES, MF_EMBEDDING_DIM).to(DEVICE)
optimizer = torch.optim.Adam(mf_model.parameters(), lr=MF_LEARNING_RATE, weight_decay=1e-6)
loss_fn = nn.BCEWithLogitsLoss()

mf_history = []

for epoch in range(1, MF_EPOCHS + 1):
    mf_model.train()

    # New negatives and a new shuffle every epoch
    negative_movies = sample_one_negative_per_user(train_users)
    order = rng_train.permutation(len(train_users))

    losses = []
    for start in range(0, len(order), MF_BATCH_SIZE):
        batch = order[start:start + MF_BATCH_SIZE]

        users = torch.tensor(train_users[batch], dtype=torch.long, device=DEVICE)
        pos_movies = torch.tensor(train_movies[batch], dtype=torch.long, device=DEVICE)
        neg_movies = torch.tensor(negative_movies[batch], dtype=torch.long, device=DEVICE)

        pos_scores = mf_model(users, pos_movies)   # should become high (label 1)
        neg_scores = mf_model(users, neg_movies)   # should become low  (label 0)

        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        loss = loss_fn(scores, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    epoch_loss = float(np.mean(losses))
    mf_history.append({"epoch": epoch, "loss": epoch_loss})
    print(f"Epoch {epoch}/{MF_EPOCHS} - loss: {epoch_loss:.4f}")

display(pd.DataFrame(mf_history))

## 23. Evaluate matrix factorization and compare baselines

In [ ]:
# Score a list of candidate movies for one user
@torch.no_grad()
def mf_score_candidates(user, candidate_movies):
    mf_model.eval()
    movies_t = torch.tensor(candidate_movies, dtype=torch.long, device=DEVICE)
    users_t = torch.full_like(movies_t, int(user))
    return mf_model(users_t, movies_t).cpu().numpy()


mf_metrics = ranking_metrics_from_scores(eval_candidates, mf_score_candidates, k=10)

baseline_results = pd.DataFrame([
    {"model": "Popularity", **pop_metrics},
    {"model": "Implicit Matrix Factorization", **mf_metrics},
])
display(baseline_results)

# Save results for later comparison with NeuMF
baseline_results.to_csv(OUTPUT_DIR / "baseline_metrics.csv", index=False)
popularity_ranked.to_csv(OUTPUT_DIR / "popularity_ranking.csv", index=False)

torch.save({
    "model_state_dict": mf_model.state_dict(),
    "num_users": NUM_USERS,
    "num_movies": NUM_MOVIES,
    "embedding_dim": MF_EMBEDDING_DIM,
    "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
    "seed": SEED,
}, OUTPUT_DIR / "matrix_factorization.pt")

print("Saved baseline metrics, popularity ranking, and MF checkpoint to:", OUTPUT_DIR.resolve())

## 24. Get recommendations from matrix factorization

This inference helper scores all movies for a known user and removes already-observed movies. For large catalogs, production systems normally pre-filter candidates or cache top recommendations rather than score the full catalog on every request.

In [ ]:
# Top-k movies for a user: score all movies, drop the ones already rated
@torch.no_grad()
def recommend_mf_for_user(user_idx, k=10):
    user_idx = int(user_idx)
    if user_idx < 0 or user_idx >= NUM_USERS:
        raise ValueError(f"Unknown user_idx {user_idx}; expected 0 to {NUM_USERS - 1}.")

    mf_model.eval()

    # Score every movie for this user
    all_movies = torch.arange(NUM_MOVIES, dtype=torch.long, device=DEVICE)
    users_t = torch.full_like(all_movies, user_idx)
    scores = mf_model(users_t, all_movies).cpu().numpy()

    result = pd.DataFrame({"movie_idx": np.arange(NUM_MOVIES), "score": scores})
    result = result[~result["movie_idx"].isin(all_seen_by_user[user_idx])]
    result = result.sort_values("score", ascending=False).head(k)

    result["movieId"] = result["movie_idx"].map(dict(zip(movie_mapping["movie_idx"], movie_mapping["movieId"])))
    result["title"] = result["movie_idx"].map(movie_title_by_idx)
    return result[["movieId", "title", "score"]].reset_index(drop=True)


# Example: any user_idx between 0 and NUM_USERS - 1
display(recommend_mf_for_user(example_user_idx, k=10))